# Kanka Character Updater (Python + Colab)

Port of the original Google Apps Script project to pure Python. This notebook preserves the same modules (config, API client, caches, formatter, renderer, image sync, orchestrator) but now runs natively inside Google Colab using the Google Docs and Drive APIs.


## Workflow rapido

1. Popola le variabili d'ambiente (`KANKA_API_TOKEN`, `CAMPAIGN_ID`, `GOOGLE_DOC_ID`) usando il blocco opzionale qui sotto oppure `os.environ`/`.env`.
2. Installa le dipendenze Colab (una sola volta per sessione).
3. Esegui `ensure_config()` e crea un `KankaCharacterUpdater`.
4. Avvia `updater.sync()` per generare il documento e, se vuoi, `updater.sync_images()` per rimpiazzare i segnaposto `{{IMAGE:type:id}}` con immagini vere.
5. Le classi sono modulari, quindi puoi anche riutilizzare `KankaAPI`, `EntityRenderer`, ecc. nei tuoi script Python.


In [23]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib requests python-dotenv beautifulsoup4 pillow tqdm


In [24]:
# Optional: store secrets when running in Colab (remove placeholder values before sharing the notebook).
import os
from google.colab import userdata
os.environ["KANKA_API_TOKEN"] = userdata.get("KANKA_API_TOKEN")
os.environ["CAMPAIGN_ID"] = userdata.get("CAMPAIGN_ID")
os.environ["GOOGLE_DOC_ID"] = userdata.get("GOOGLE_DOC_ID")
os.environ["TINYPNG_API_KEY"] = userdata.get("TINYPNG_API_KEY")  # facoltativo

In [25]:
import base64
import io
import json
import logging
import math
import os
import re
import time
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Dict, Iterable, List, Optional, Tuple

import google.auth
import requests
from bs4 import BeautifulSoup
from PIL import Image
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseUpload
from tqdm.auto import tqdm

try:
    from google.colab import auth as colab_auth  # type: ignore
    IN_COLAB = True
except ModuleNotFoundError:  # pragma: no cover
    colab_auth = None
    IN_COLAB = False

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")

KANKA_API_BASE = "https://kanka.io/api/1.0"
MAX_RETRIES = 3
INITIAL_RETRY_DELAY = 1.5
SUMMARY_PLACEHOLDER = "[[KANKA_SYNC_SUMMARY]]"
SUPPORTED_ENTITY_TYPES = [
    "characters",
    "locations",
    "families",
    "organisations",
    "races",
    "creatures",
    "items",
    "events",
    "journals",
    "quests",
    "abilities",
    "calendars",
    "timelines",
]
REFERENCE_ENTITY_TYPES = [
    "locations",
    "families",
    "organisations",
    "races",
    "creatures",
    "items",
    "events",
    "journals",
    "quests",
    "abilities",
    "calendars",
    "timelines",
    "tags",
]
POINTS_PER_INCH = 72
DEFAULT_IMAGE_MAX_DIMENSION = 1024
DEFAULT_IMAGE_WIDTH_INCHES = 3.2
REFERENCE_PATTERN = re.compile(r"\[([a-z_]+):(\d+)\]", re.IGNORECASE)
IMAGE_PLACEHOLDER_PATTERN = re.compile(r"\{\{IMAGE:([a-z_]+):(\d+)\}\}", re.IGNORECASE)
SCOPES = [
    "https://www.googleapis.com/auth/documents",
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/drive.file",
]


In [26]:
@dataclass
class KankaConfig:
    api_token: str = ""
    campaign_id: str = ""
    document_id: str = ""
    tinypng_api_key: Optional[str] = None
    max_image_dimension: int = DEFAULT_IMAGE_MAX_DIMENSION
    use_drive_image_staging: bool = True

    @property
    def is_ready(self) -> bool:
        return bool(self.api_token and self.campaign_id and self.document_id)


def load_config_from_env() -> KankaConfig:
    def _read_int(value: Optional[str], fallback: int) -> int:
        try:
            return int(value) if value else fallback
        except (TypeError, ValueError):
            return fallback

    return KankaConfig(
        api_token=os.getenv("KANKA_API_TOKEN", "").strip(),
        campaign_id=os.getenv("CAMPAIGN_ID", "").strip(),
        document_id=os.getenv("GOOGLE_DOC_ID", "").strip(),
        tinypng_api_key=(os.getenv("TINYPNG_API_KEY", "").strip() or None),
        max_image_dimension=_read_int(os.getenv("TINYPNG_MAX_DIMENSION"), DEFAULT_IMAGE_MAX_DIMENSION),
    )


def ensure_config(config: Optional[KankaConfig] = None) -> KankaConfig:
    cfg = config or load_config_from_env()
    for field_name in ("api_token", "campaign_id", "document_id"):
        if not getattr(cfg, field_name):
            prompt = field_name.replace("_", " ").upper()
            setattr(cfg, field_name, input(f"Inserisci {prompt}: ").strip())
    if not cfg.max_image_dimension:
        cfg.max_image_dimension = DEFAULT_IMAGE_MAX_DIMENSION
    return cfg


In [27]:
def build_google_services():
    if IN_COLAB and colab_auth is not None:
        colab_auth.authenticate_user()
    creds, _ = google.auth.default(scopes=SCOPES)
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
    docs_service = build("docs", "v1", credentials=creds)
    drive_service = build("drive", "v3", credentials=creds)
    return docs_service, drive_service


In [28]:
class KankaAPI:
    def __init__(self, config: KankaConfig):
        self.config = config
        self.base_url = f"{KANKA_API_BASE}/campaigns/{config.campaign_id}"
        self.session = requests.Session()
        self.session.headers.update(
            {
                "Authorization": f"Bearer {config.api_token}",
                "Accept": "application/json",
                "Content-Type": "application/json",
            }
        )
        self.cache: Dict[str, Any] = {}

    def _request(self, url: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        delay = INITIAL_RETRY_DELAY
        last_error: Optional[Exception] = None
        for attempt in range(MAX_RETRIES):
            try:
                response = self.session.get(url, params=params, timeout=60)
                if response.status_code == 429:
                    wait = int(response.headers.get("Retry-After", 2))
                    logging.warning("Rate limit: attendo %ss", wait)
                    time.sleep(wait)
                    continue
                if response.status_code >= 500:
                    raise RuntimeError(f"HTTP {response.status_code}: {response.text[:200]}")
                if not response.ok:
                    raise RuntimeError(f"HTTP {response.status_code}: {response.text[:200]}")
                return response.json() if response.text else {}
            except Exception as exc:  # pragma: no cover
                last_error = exc
                logging.warning("Tentativo %s fallito: %s", attempt + 1, exc)
                time.sleep(delay)
                delay *= 2
        raise RuntimeError(f"Kanka request failed after {MAX_RETRIES} attempts: {last_error}")

    def _paginate(self, endpoint: str, params: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        url = f"{self.base_url}/{endpoint}" if not endpoint.startswith("http") else endpoint
        results: List[Dict[str, Any]] = []
        while url:
            payload = self._request(url, params=params)
            data = payload.get("data") or []
            if not isinstance(data, list):
                data = [data]
            results.extend(data)
            links = payload.get("links") or {}
            url = links.get("next")
        return results

    def get_entities(self, entity_type: str, force_refresh: bool = False) -> List[Dict[str, Any]]:
        key = f"{entity_type}:list"
        if not force_refresh and key in self.cache:
            return self.cache[key]
        entities = self._paginate(entity_type)
        self.cache[key] = entities
        return entities

    def get_entity(self, entity_type: str, entity_id: int) -> Optional[Dict[str, Any]]:
        key = f"{entity_type}:{entity_id}"
        if key in self.cache:
            return self.cache[key]
        try:
            data = self._request(f"{self.base_url}/{entity_type}/{entity_id}").get("data")
        except RuntimeError:
            data = None
        if not data:
            try:
                data = self._request(f"{self.base_url}/entities/{entity_id}").get("data")
            except RuntimeError:
                data = None
        self.cache[key] = data
        return data

    def fetch_entity_by_id(self, entity_id: int) -> Optional[Dict[str, Any]]:
        key = f"entities:{entity_id}"
        if key in self.cache:
            return self.cache[key]
        try:
            data = self._request(f"https://kanka.io/api/1.0/entities/{entity_id}").get("data")
        except RuntimeError:
            data = None
        self.cache[key] = data
        return data


class EntityCache:
    ALIASES = {
        "character": "characters",
        "creature": "creatures",
        "location": "locations",
        "family": "families",
        "organisation": "organisations",
        "organization": "organisations",
        "race": "races",
        "tag": "tags",
        "item": "items",
        "event": "events",
        "journal": "journals",
        "quest": "quests",
        "ability": "abilities",
        "calendar": "calendars",
        "timeline": "timelines",
        "note": "journals",
        "entity": "entities",
    }

    def __init__(self, api: KankaAPI):
        self.api = api
        self.maps: Dict[str, Dict[str, str]] = {}

    def _normalize(self, type_name: Optional[str]) -> Optional[str]:
        if not type_name:
            return None
        key = str(type_name).strip().lower()
        return self.ALIASES.get(key, key)

    def remember_many(self, type_name: str, entities: Iterable[Dict[str, Any]]):
        norm = self._normalize(type_name)
        if not norm:
            return
        self.maps.setdefault(norm, {})
        for entity in entities:
            if not entity:
                continue
            entity_id = entity.get("id")
            name = entity.get("name")
            if entity_id and name:
                self.maps[norm][str(entity_id)] = str(name)

    def resolve(self, type_name: str, entity_id: Any) -> Optional[str]:
        norm = self._normalize(type_name)
        if not norm or entity_id is None:
            return None
        key = str(entity_id)
        value = self.maps.get(norm, {}).get(key)
        if value:
            return value
        fetched = self.api.get_entity(norm, int(entity_id)) if norm != "entities" else self.api.fetch_entity_by_id(int(entity_id))
        if fetched and fetched.get("name"):
            self.maps.setdefault(norm, {})[key] = fetched["name"]
            return fetched["name"]
        return None

    def preload(self, types: Iterable[str]):
        for type_name in types:
            try:
                entities = self.api.get_entities(type_name)
                self.remember_many(type_name, entities)
                logging.info("Precaricati %s elementi per %s", len(entities), type_name)
            except Exception as exc:
                logging.warning("Impossibile precaricare %s: %s", type_name, exc)


In [29]:
def strip_html(value: Optional[str]) -> str:
    if not value:
        return ""
    soup = BeautifulSoup(value, "html.parser")
    return soup.get_text(separator=" ", strip=True)


def resolve_references(value: Optional[str], cache: EntityCache) -> str:
    if not value:
        return ""

    def _replace(match: re.Match[str]) -> str:
        entity_type, entity_id = match.groups()
        resolved = cache.resolve(entity_type, entity_id)
        return resolved or match.group(0)

    return REFERENCE_PATTERN.sub(_replace, value)


def clean_text(value: Optional[str], cache: EntityCache, allow_html: bool = False) -> str:
    parsed = resolve_references(value, cache)
    return parsed if allow_html else strip_html(parsed)


def format_datetime(value: Optional[str]) -> str:
    if not value:
        return ""
    try:
        dt = datetime.fromisoformat(value.replace("Z", "+00:00"))
        return dt.strftime("%d/%m/%Y %H:%M")
    except ValueError:
        return value


def create_image_placeholder(entity_type: str, entity_id: Any) -> str:
    return f"{{{{IMAGE:{entity_type}:{entity_id}}}}}"


def parse_image_placeholder(text: str) -> Optional[Tuple[str, str]]:
    if not text:
        return None
    match = IMAGE_PLACEHOLDER_PATTERN.search(text.strip())
    if not match:
        return None
    return match.group(1), match.group(2)


In [30]:
class GoogleDocWriter:
    def __init__(self, docs_service, document_id: str):
        self.docs = docs_service
        self.document_id = document_id
        self.cursor = self._fetch_cursor()

    def _fetch_cursor(self) -> int:
        doc = self.docs.documents().get(documentId=self.document_id).execute()
        content = doc.get("body", {}).get("content", [])
        if not content:
            return 1
        return max(1, content[-1].get("endIndex", 1) - 1)

    def _execute(self, requests: List[Dict[str, Any]]):
        if not requests:
            return
        self.docs.documents().batchUpdate(documentId=self.document_id, body={"requests": requests}).execute()

    def refresh_cursor(self):
        self.cursor = self._fetch_cursor()

    def clear(self):
        doc = self.docs.documents().get(documentId=self.document_id).execute()
        end_index = doc.get("body", {}).get("content", [{}])[-1].get("endIndex", 1)
        if end_index <= 1:
            self.cursor = 1
            return
        self._execute(
            [
                {
                    "deleteContentRange": {
                        "range": {
                            "startIndex": 1,
                            "endIndex": end_index - 1,
                        }
                    }
                }
            ]
        )
        self.cursor = 1

    def _insert_text(self, text: str, index: Optional[int] = None) -> Tuple[int, int]:
        if text == "":
            return self.cursor, self.cursor
        location = index if index is not None else self.cursor
        requests = [
            {
                "insertText": {
                    "location": {"index": location},
                    "text": text,
                }
            }
        ]
        self._execute(requests)
        length = len(text)
        if index is None:
            self.cursor += length
        else:
            self.cursor = max(self.cursor, index + length)
        return location, location + length

    def append_paragraph(self, text: str = ""):
        if not text.endswith("\n"):
            text += "\n"
        self._insert_text(text)

    def append_heading(self, text: str, level: int = 1):
        if not text.endswith("\n"):
            text += "\n"
        start, end = self._insert_text(text)
        named_style = {1: "HEADING_1", 2: "HEADING_2", 3: "HEADING_3", 4: "HEADING_4"}.get(level, "NORMAL_TEXT")
        self._execute(
            [
                {
                    "updateParagraphStyle": {
                        "range": {"startIndex": start, "endIndex": end},
                        "paragraphStyle": {"namedStyleType": named_style},
                        "fields": "namedStyleType",
                    }
                }
            ]
        )

    def append_key_value_pairs(self, pairs: List[Tuple[str, str]], heading: Optional[str] = None, level: int = 3):
        if heading:
            self.append_heading(heading, level)
        if not pairs:
            return
        lines = [f"{key}: {value}" for key, value in pairs]
        text = "\n".join(lines) + "\n"
        start, end = self._insert_text(text)
        self._execute(
            [
                {
                    "createParagraphBullets": {
                        "range": {"startIndex": start, "endIndex": end},
                        "bulletPreset": "BULLET_DISC_CIRCLE_SQUARE",
                    }
                }
            ]
        )

    def append_bullet_list(self, items: List[str]):
        if not items:
            return
        text = "\n".join(items) + "\n"
        start, end = self._insert_text(text)
        self._execute(
            [
                {
                    "createParagraphBullets": {
                        "range": {"startIndex": start, "endIndex": end},
                        "bulletPreset": "BULLET_DISC_CIRCLE_SQUARE",
                    }
                }
            ]
        )

    def append_image_placeholder(self, entity_type: str, entity_id: Any):
        placeholder = create_image_placeholder(entity_type, entity_id)
        if not placeholder.endswith("\n"):
            placeholder += "\n"
        start, end = self._insert_text(placeholder)
        self._execute(
            [
                {
                    "updateTextStyle": {
                        "range": {"startIndex": start, "endIndex": end},
                        "textStyle": {
                            "italic": True,
                            "foregroundColor": {
                                "color": {"rgbColor": {"red": 0.5, "green": 0.5, "blue": 0.5}}
                            },
                        },
                        "fields": "italic,foregroundColor",
                    }
                }
            ]
        )

    def _find_text_range(self, needle: str) -> Optional[Tuple[int, int]]:
        doc = self.docs.documents().get(documentId=self.document_id).execute()
        for element in doc.get("body", {}).get("content", []):
            paragraph = element.get("paragraph")
            if not paragraph:
                continue
            for child in paragraph.get("elements", []):
                text_run = child.get("textRun")
                if not text_run:
                    continue
                content = text_run.get("content", "")
                if needle in content:
                    start = child.get("startIndex", 1) + content.index(needle)
                    end = start + len(needle)
                    return start, end
        return None

    def insert_summary(self, counts: Dict[str, int]):
        summary_lines = [f"{entity_type.capitalize()}: {counts.get(entity_type, 0)}" for entity_type in SUPPORTED_ENTITY_TYPES]
        block = "Sommario\n" + "\n".join(summary_lines) + "\n\n"
        target = self._find_text_range(SUMMARY_PLACEHOLDER)
        if not target:
            self.append_heading("Sommario", 2)
            self.append_bullet_list(summary_lines)
            return
        start, end = target
        requests: List[Dict[str, Any]] = [
            {"deleteContentRange": {"range": {"startIndex": start, "endIndex": end}}},
            {"insertText": {"location": {"index": start}, "text": block}},
            {
                "updateParagraphStyle": {
                    "range": {"startIndex": start, "endIndex": start + len("Sommario\n")},
                    "paragraphStyle": {"namedStyleType": "HEADING_2"},
                    "fields": "namedStyleType",
                }
            },
        ]
        bullet_start = start + len("Sommario\n")
        bullet_end = start + len(block)
        requests.append(
            {
                "createParagraphBullets": {
                    "range": {"startIndex": bullet_start, "endIndex": bullet_end},
                    "bulletPreset": "BULLET_DISC_CIRCLE_SQUARE",
                }
            }
        )
        self._execute(requests)
        self.refresh_cursor()

In [31]:
class EntityRenderer:
    def __init__(self, writer: GoogleDocWriter, cache: EntityCache):
        self.writer = writer
        self.cache = cache
    def render(self, entity_type: str, entity: Dict[str, Any]):
        if entity_type == "characters":
            self._render_character(entity)
        elif entity_type == "calendars":
            self._render_calendar(entity)
        elif entity_type == "timelines":
            self._render_timeline(entity)
        else:
            self._render_generic(entity_type, entity)
        self.writer.append_paragraph()
    def _render_character(self, entity: Dict[str, Any]):
        title = entity.get("name") or "(senza nome)"
        if entity.get("is_dead"):
            title += " (Deceduto/a)"
        self.writer.append_heading(title, 2)
        if entity.get("image_full") or entity.get("image_thumb"):
            self.writer.append_image_placeholder("characters", entity.get("id"))
        metadata = self._build_metadata(entity)
        if metadata:
            self.writer.append_key_value_pairs(metadata, heading="Metadati", level=3)
        main_info = []
        for key, label in (
            ("title", "Titolo"),
            ("type", "Tipo"),
            ("age", "Eta"),
            ("sex", "Sesso"),
            ("pronouns", "Pronomi"),
        ):
            value = entity.get(key)
            if value:
                main_info.append((label, str(value)))
        race_names = self._resolve_many("races", entity)
        if race_names:
            main_info.append(("Razze", ", ".join(race_names)))
        family_names = self._resolve_many("families", entity)
        if family_names:
            main_info.append(("Famiglie", ", ".join(family_names)))
        location_name = self._resolve_location(entity)
        if location_name:
            main_info.append(("Posizione", location_name))
        if main_info:
            self.writer.append_key_value_pairs(main_info, heading="Informazioni principali", level=3)
        biography = clean_text(entity.get("entry"), self.cache)
        if biography:
            self.writer.append_heading("Biografia", 3)
            self.writer.append_paragraph(biography)
        appearance = clean_text(entity.get("appearance"), self.cache)
        if appearance:
            self.writer.append_heading("Aspetto fisico", 3)
            self.writer.append_paragraph(appearance)
        history = clean_text(entity.get("history"), self.cache)
        if history:
            self.writer.append_heading("Storia", 3)
            self.writer.append_paragraph(history)
        self._render_attributes(entity)
        self._render_personality(entity)
        self._render_relations(entity)
        self._render_inventory(entity)
        self._render_notes(entity)
    def _render_generic(self, entity_type: str, entity: Dict[str, Any]):
        name = entity.get("name") or "(senza nome)"
        self.writer.append_heading(name, 2)
        if entity.get("image_full") or entity.get("image_thumb"):
            self.writer.append_image_placeholder(entity_type, entity.get("id"))
        metadata = self._build_metadata(entity)
        if metadata:
            self.writer.append_key_value_pairs(metadata, heading="Metadati", level=3)
        info_pairs: List[Tuple[str, str]] = []
        for key, label in (
            ("title", "Titolo"),
            ("type", "Tipo"),
            ("status", "Stato"),
            ("age", "Eta"),
        ):
            value = entity.get(key)
            if value:
                info_pairs.append((label, str(value)))
        location_name = self._resolve_location(entity)
        if location_name:
            info_pairs.append(("Posizione", location_name))
        family_names = self._resolve_many("families", entity)
        if family_names:
            info_pairs.append(("Famiglie", ", ".join(family_names)))
        race_names = self._resolve_many("races", entity)
        if race_names:
            info_pairs.append(("Razze", ", ".join(race_names)))
        if info_pairs:
            self.writer.append_key_value_pairs(info_pairs, heading="Informazioni principali", level=3)
        description = clean_text(entity.get("entry"), self.cache)
        if description:
            self.writer.append_heading("Descrizione", 3)
            self.writer.append_paragraph(description)
        attributes = entity.get("attributes") or []
        if isinstance(attributes, list) and attributes:
            pairs = []
            for attr in attributes:
                name = attr.get("name") or attr.get("api_name")
                value = attr.get("value")
                if name and value not in (None, ""):
                    pairs.append((name, str(value)))
            if pairs:
                self.writer.append_key_value_pairs(pairs, heading="Attributi", level=3)
        tags = self._format_tags(entity)
        if tags:
            self.writer.append_heading("Tags", 3)
            self.writer.append_bullet_list(tags)
    def _render_calendar(self, entity: Dict[str, Any]):
        name = entity.get("name") or "Calendario"
        self.writer.append_heading(name, 2)
        if entity.get("image_full"):
            self.writer.append_image_placeholder("calendars", entity.get("id"))
        if entity.get("date"):
            self.writer.append_paragraph(f"Data di riferimento: {entity['date']}")
        months = entity.get("months") or []
        if months:
            lines = [f"{month.get('name')} ({month.get('length')} giorni)" for month in months]
            self.writer.append_heading("Mesi dell'anno", 3)
            self.writer.append_bullet_list(lines)
        weekdays = entity.get("weekdays") or []
        if weekdays:
            self.writer.append_heading("Giorni della settimana", 3)
            self.writer.append_bullet_list([str(day) for day in weekdays])
        seasons = entity.get("seasons") or []
        if seasons:
            self.writer.append_heading("Stagioni", 3)
            formatted = [
                f"{season.get('name')} - giorno {season.get('day')} del mese {season.get('month')}"
                for season in seasons
            ]
            self.writer.append_bullet_list(formatted)
        moons = entity.get("moons") or []
        if moons:
            self.writer.append_heading("Lune", 3)
            moon_lines = [
                f"{moon.get('name')} - piena ogni {moon.get('fullmoon')} giorni"
                for moon in moons
            ]
            self.writer.append_bullet_list(moon_lines)
    def _render_timeline(self, entity: Dict[str, Any]):
        name = entity.get("name") or "Timeline"
        self.writer.append_heading(name, 2)
        if entity.get("image_full"):
            self.writer.append_image_placeholder("timelines", entity.get("id"))
        eras = entity.get("eras") or []
        for era in eras:
            title = era.get("name") or "Era"
            if era.get("abbreviation"):
                title += f" ({era['abbreviation']})"
            self.writer.append_heading(title, 3)
            period = []
            if era.get("start_year") is not None:
                period.append(str(era.get("start_year")))
            if era.get("end_year") is not None:
                period.append(str(era.get("end_year")))
            if period:
                self.writer.append_paragraph("Periodo: " + " - ".join(period))
            entry = clean_text(era.get("entry"), self.cache)
            if entry:
                self.writer.append_paragraph(entry)
            elements = sorted(era.get("elements") or [], key=lambda e: e.get("position", 0))
            if elements:
                lines = []
                for element in elements:
                    label = element.get("name") or "Evento"
                    date = element.get("date")
                    entry_text = clean_text(element.get("entry"), self.cache)
                    line = f"{label} ({date})" if date else label
                    lines.append(line)
                    if entry_text:
                        self.writer.append_paragraph(entry_text)
                self.writer.append_bullet_list(lines)
    def _build_metadata(self, entity: Dict[str, Any]) -> List[Tuple[str, str]]:
        pairs = []
        for key, label in (
            ("id", "ID"),
            ("type", "Tipo"),
            ("slug", "Slug"),
            ("created_at", "Creato"),
            ("updated_at", "Modificato"),
            ("created_by_name", "Creato da"),
            ("owner_name", "Proprietario"),
        ):
            value = entity.get(key)
            if value:
                if "_at" in key:
                    value = format_datetime(value)
                pairs.append((label, str(value)))
        return pairs
    def _resolve_many(self, collection_type: str, entity: Dict[str, Any]) -> List[str]:
        values: List[str] = []
        direct_name = entity.get(f"{collection_type[:-1]}_name")
        if direct_name:
            values.append(str(direct_name))
        singular_id = entity.get(f"{collection_type[:-1]}_id")
        if singular_id and not direct_name:
            resolved = self.cache.resolve(collection_type, singular_id)
            if resolved:
                values.append(resolved)
        ids = entity.get(collection_type) or []
        if isinstance(ids, list):
            for item in ids:
                resolved = self.cache.resolve(collection_type, item)
                if resolved:
                    values.append(resolved)
        private_ids = entity.get(f"private_{collection_type}") or []
        if isinstance(private_ids, list):
            for item in private_ids:
                resolved = self.cache.resolve(collection_type, item)
                if resolved:
                    values.append(f"{resolved} (privato)")
        return values
    def _resolve_location(self, entity: Dict[str, Any]) -> Optional[str]:
        if entity.get("location_name"):
            return entity["location_name"]
        if entity.get("location_id"):
            return self.cache.resolve("locations", entity["location_id"])
        return None
    def _render_attributes(self, entity: Dict[str, Any]):
        attributes = entity.get("attributes") or []
        if not isinstance(attributes, list) or not attributes:
            return
        sections: Dict[str, List[Tuple[str, str]]] = {}
        for attr in attributes:
            name = attr.get("name") or attr.get("api_name")
            value = attr.get("value")
            if not name or value in (None, ""):
                continue
            section = attr.get("section") or "Generali"
            sections.setdefault(section, []).append((name, str(value)))
        for section, pairs in sections.items():
            self.writer.append_key_value_pairs(pairs, heading=f"Attributi - {section}", level=3)
    def _render_personality(self, entity: Dict[str, Any]):
        traits = entity.get("traits") or entity.get("personality") or []
        tag_lines = self._format_tags(entity)
        if not traits and not tag_lines:
            return
        self.writer.append_heading("Personalita", 3)
        trait_lines = []
        for trait in traits:
            if isinstance(trait, dict):
                label = trait.get("name") or trait.get("title") or "Tratto"
                value = clean_text(trait.get("entry"), self.cache)
                if value:
                    trait_lines.append(f"{label}: {value}")
            elif isinstance(trait, str):
                trait_lines.append(trait)
        self.writer.append_bullet_list(trait_lines + tag_lines)
    def _format_tags(self, entity: Dict[str, Any]) -> List[str]:
        tags = entity.get("tags") or []
        lines = []
        for tag in tags:
            if isinstance(tag, dict) and tag.get("name"):
                lines.append(str(tag["name"]))
            elif isinstance(tag, str):
                lines.append(tag)
        if entity.get("tag_list"):
            lines.append(str(entity["tag_list"]))
        return lines
    def _render_relations(self, entity: Dict[str, Any]):
        relations = entity.get("relations") or []
        if not relations:
            return
        self.writer.append_heading("Relazioni", 3)
        lines = []
        for relation in relations:
            target = relation.get("target_name")
            if not target and relation.get("target_id"):
                target_type = relation.get("target_entity_type") or "characters"
                target = self.cache.resolve(target_type, relation["target_id"]) or "[sconosciuto]"
            label = relation.get("relation") or relation.get("type") or "Relazione"
            attitude = relation.get("attitude")
            detail = f"{label} con {target}"
            if attitude:
                detail += f" ({attitude})"
            lines.append(detail)
        self.writer.append_bullet_list(lines)
    def _render_inventory(self, entity: Dict[str, Any]):
        inventory = entity.get("inventory") or []
        if not inventory:
            return
        self.writer.append_heading("Inventario", 3)
        lines = []
        for item in inventory:
            name = item.get("name") or item.get("entity", {}).get("name") or "Oggetto"
            amount = f" x{item['amount']}" if item.get("amount") else ""
            position = f" [{item['position']}]" if item.get("position") else ""
            notes = clean_text(item.get("notes"), self.cache)
            suffix = f" - {notes}" if notes else ""
            lines.append(f"{name}{amount}{position}{suffix}")
        self.writer.append_bullet_list(lines)
    def _render_notes(self, entity: Dict[str, Any]):
        notes = clean_text(entity.get("private_notes"), self.cache)
        visibility_flags = []
        if entity.get("is_private"):
            visibility_flags.append("Privato")
        if entity.get("is_personality_visible") is False:
            visibility_flags.append("Personalita nascosta")
        if entity.get("is_attributes_private"):
            visibility_flags.append("Attributi privati")
        if notes or visibility_flags:
            self.writer.append_heading("Note", 3)
            if notes:
                self.writer.append_paragraph(notes)
            if visibility_flags:
                self.writer.append_bullet_list(visibility_flags)


In [32]:
class DriveImageStagingArea:
    def __init__(self, drive_service, folder_name: str = "KankaCharacterUpdater"):
        self.drive = drive_service
        self.folder_name = folder_name
        self.folder_id = self._ensure_folder()
        self.created_files: List[str] = []

    def _ensure_folder(self) -> Optional[str]:
        if not self.drive:
            return None
        query = " and ".join(
            [
                "mimeType='application/vnd.google-apps.folder'",
                f"name='{self.folder_name}'",
                "trashed=false",
            ]
        )
        existing = self.drive.files().list(q=query, fields="files(id,name)", pageSize=1).execute().get("files", [])
        if existing:
            return existing[0]["id"]
        metadata = {"name": self.folder_name, "mimeType": "application/vnd.google-apps.folder"}
        created = self.drive.files().create(body=metadata, fields="id").execute()
        return created.get("id")

    def upload(self, filename: str, data: bytes, mime_type: str) -> Optional[str]:
        if not (self.drive and self.folder_id):
            return None
        media = MediaIoBaseUpload(io.BytesIO(data), mimetype=mime_type, resumable=False)
        metadata = {"name": filename, "parents": [self.folder_id]}
        created = self.drive.files().create(body=metadata, media_body=media, fields="id").execute()
        file_id = created.get("id")
        self.drive.permissions().create(fileId=file_id, body={"type": "anyone", "role": "reader"}).execute()
        self.created_files.append(file_id)
        return f"https://drive.google.com/uc?export=view&id={file_id}"

    def cleanup(self):
        if not self.drive:
            return
        for file_id in list(self.created_files):
            try:
                self.drive.files().delete(fileId=file_id).execute()
            except HttpError:
                pass
            finally:
                self.created_files.remove(file_id)


class ImageSynchronizer:
    def __init__(self, docs_service, drive_service, api: KankaAPI, config: KankaConfig):
        self.docs = docs_service
        self.api = api
        self.config = config
        self.staging = DriveImageStagingArea(drive_service) if config.use_drive_image_staging else None

    def sync_placeholders(self):
        doc = self.docs.documents().get(documentId=self.config.document_id).execute()
        placeholders = self._collect_placeholders(doc)
        if not placeholders:
            logging.info("Nessun segnaposto immagine trovato.")
            return
        inserted = 0
        skipped = 0
        for placeholder in tqdm(placeholders, desc="Immagini"):
            entity = self.api.get_entity(placeholder["type"], int(placeholder["id"]))
            if not entity:
                skipped += 1
                continue
            image_bytes, mime_type = self._download_image(entity)
            if not image_bytes:
                skipped += 1
                continue
            public_url = None
            if self.staging:
                filename = f"kcu-{placeholder['type']}-{placeholder['id']}.png"
                public_url = self.staging.upload(filename, image_bytes, mime_type)
            else:
                public_url = entity.get("image_full")
            if not public_url:
                skipped += 1
                continue
            self._replace_with_image(placeholder, public_url)
            inserted += 1
        logging.info("Sincronizzazione immagini: %s inserite, %s senza immagine", inserted, skipped)

    def _collect_placeholders(self, doc: Dict[str, Any]) -> List[Dict[str, Any]]:
        results: List[Dict[str, Any]] = []
        for element in doc.get("body", {}).get("content", []):
            paragraph = element.get("paragraph")
            if not paragraph:
                continue
            for child in paragraph.get("elements", []):
                text_run = child.get("textRun")
                if not text_run:
                    continue
                text = text_run.get("content", "").strip()
                parsed = parse_image_placeholder(text)
                if parsed:
                    entity_type, entity_id = parsed
                    results.append(
                        {
                            "type": entity_type,
                            "id": entity_id,
                            "start": child.get("startIndex", 1),
                            "end": child.get("endIndex", child.get("startIndex", 1) + len(text)),
                        }
                    )
        return results

    def _download_image(self, entity: Dict[str, Any]) -> Tuple[Optional[bytes], str]:
        url = entity.get("image_full") or entity.get("image_thumb")
        if not url:
            return None, ""
        response = requests.get(url, timeout=60)
        if response.status_code >= 400:
            return None, ""
        image = Image.open(io.BytesIO(response.content))
        image = self._resize_image(image)
        buffer = io.BytesIO()
        image.save(buffer, format="PNG")
        return buffer.getvalue(), "image/png"

    def _resize_image(self, image: Image.Image) -> Image.Image:
        max_dim = max(1, self.config.max_image_dimension)
        width, height = image.size
        scale = min(1.0, max_dim / float(max(width, height)))
        if scale >= 1.0:
            return image
        new_size = (int(width * scale), int(height * scale))
        return image.resize(new_size, Image.LANCZOS)

    def _replace_with_image(self, placeholder: Dict[str, Any], image_url: str):
        start = placeholder["start"]
        end = placeholder["end"]
        width_points = DEFAULT_IMAGE_WIDTH_INCHES * POINTS_PER_INCH
        requests = [
            {"deleteContentRange": {"range": {"startIndex": start, "endIndex": end}}},
            {
                "insertInlineImage": {
                    "location": {"index": start},
                    "uri": image_url,
                    "objectSize": {
                        "height": {"magnitude": width_points, "unit": "PT"},
                        "width": {"magnitude": width_points, "unit": "PT"},
                    },
                }
            },
        ]
        self.docs.documents().batchUpdate(documentId=self.config.document_id, body={"requests": requests}).execute()


In [33]:
class KankaCharacterUpdater:
    def __init__(self, config: Optional[KankaConfig] = None):
        self.config = ensure_config(config)
        self.docs_service = None
        self.drive_service = None
        self.api: Optional[KankaAPI] = None
        self.cache: Optional[EntityCache] = None
        self.writer: Optional[GoogleDocWriter] = None
        self.renderer: Optional[EntityRenderer] = None
        self.image_sync: Optional[ImageSynchronizer] = None

    def _bootstrap(self):
        if self.docs_service and self.api and self.cache and self.writer and self.renderer:
            return
        self.docs_service, self.drive_service = build_google_services()
        self.api = KankaAPI(self.config)
        self.cache = EntityCache(self.api)
        self.writer = GoogleDocWriter(self.docs_service, self.config.document_id)
        self.renderer = EntityRenderer(self.writer, self.cache)
        self.image_sync = ImageSynchronizer(self.docs_service, self.drive_service, self.api, self.config)

    def preload_references(self):
        assert self.cache is not None
        assert self.api is not None
        self.cache.preload(REFERENCE_ENTITY_TYPES)

    def sync(self, entity_types: Optional[List[str]] = None):
        self._bootstrap()
        assert self.writer and self.api and self.cache and self.renderer
        entity_types = entity_types or SUPPORTED_ENTITY_TYPES
        self.writer.clear()
        self.writer.append_heading("Enciclopedia Kanka", 1)
        self.writer.append_paragraph(f"Aggiornato il: {datetime.utcnow().strftime('%d/%m/%Y %H:%M')} UTC")
        self.writer.append_paragraph()
        self.writer.append_paragraph(SUMMARY_PLACEHOLDER)
        self.writer.append_paragraph()
        self.preload_references()
        counts: Dict[str, int] = {}
        for entity_type in entity_types:
            logging.info("Sincronizzo %s...", entity_type)
            entities = self.api.get_entities(entity_type)
            counts[entity_type] = len(entities)
            self.cache.remember_many(entity_type, entities)
            if not entities:
                continue
            self.writer.append_heading(entity_type.capitalize(), 1)
            for entity in tqdm(entities, desc=entity_type.capitalize()):
                self.renderer.render(entity_type, entity)
        self.writer.insert_summary(counts)
        logging.info("Documento aggiornato correttamente.")

    def sync_images(self):
        self._bootstrap()
        assert self.image_sync is not None
        self.image_sync.sync_placeholders()


In [34]:
# Esempio di utilizzo
# config = ensure_config()
# updater = KankaCharacterUpdater(config)
# updater.sync()  # oppure updater.sync(["characters", "locations"])
# updater.sync_images()  # opzionale, dopo aver generato i segnaposto
